# 11 — Fine-grained finance dimensions and electoral performance (2024)

This notebook answers the **second half of Workplan Question 3**:

> Where we have more fine-grained data, look at specific dimensions of
> donation/spending data versus first-place votes and ballot mentions.

Notebook 10 established that **how much** money a campaign had is related to
support. Here we ask whether **how that money is structured** carries extra
information: many small donors versus a few large ones, many small expenditures
versus a few big media buys.

This matters beyond Question 3. These same five contribution-size shares are the
dimensions we will later use to group candidates into similar finance profiles,
so getting an intuition for them now pays off twice.

The notebook stays deliberately simple: one candidate per row, a short list of
interpretable features, one variable at a time, plus interactive figures to see
which candidates sit behind each coefficient.

**These are screening associations, not causal effects.**


## What counts as "fine-grained" here?

Both fundraising and spending were split into five size bins upstream:

| Bin | Amount |
|---|---:|
| Micro | up to $25 |
| Small | $25 to $100 |
| Medium | $100 to $250 |
| Large | $250 to $1,000 |
| Mega | over $1,000 |

For each candidate we then have, in each bin, the **share of dollars** and the
**share of records**. Those two answer different questions: dollar share asks
"where did the money come from?", record share asks "how many people or
transactions were involved?" A candidate can raise most of their *dollars* from
Mega gifts while most of their *donations* are Micro.

Plus three simpler measures: record count, average, and median.

**One caveat on the spending bins:** they describe transaction **size**, not
purpose. A $40,000 expenditure lands in Mega whether it was television
advertising or a staff payroll run. So spending-bin results tell us about
transaction structure, not strategy.


## 1. Setup

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import statsmodels.api as sm
from IPython.display import display

pd.set_option("display.max_columns", 140)
pd.set_option("display.width", 190)

# ------------------------------------------------------------
# Find the repository root, so this notebook runs from either
# the repo root or the notebooks/ folder.
# ------------------------------------------------------------

cwd = Path.cwd().resolve()

if (cwd / "pyproject.toml").exists():
    ROOT = cwd
elif (cwd.parent / "pyproject.toml").exists():
    ROOT = cwd.parent
else:
    raise FileNotFoundError(
        "Could not find the repository root. "
        "Expected pyproject.toml in the current directory or its parent."
    )

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from helpers.paths import (
    PROCESSED,
    fundraising_processed_dir,
    spending_processed_dir,
)

YEAR = 2024
CONTEST = "city_council"

print("ROOT:", ROOT)


ROOT: /Users/marcy/mggg/projects/Portland/portland-fundraising-support-analysis


## 2. Load the profile tables

We use the **wide** profile files, which have one row per candidate and one
column per bin-and-metric combination.

Both wide files carry columns with the same names (for example
`canonical_candidate`), so pandas needs suffixes to keep them apart. We give the
spending file the `_spending` suffix; anything from ballot support keeps its
original name.


In [2]:
support_path = PROCESSED / "ballot_support" / str(YEAR) / "candidate_ballot_support_2024.csv"
fundraising_wide_path = fundraising_processed_dir(YEAR, CONTEST) / "openelections_candidate_fundraising_profiles_wide.csv"
spending_wide_path = spending_processed_dir(YEAR, CONTEST) / "orestar_candidate_spending_profiles_wide.csv"

support = pd.read_csv(support_path, low_memory=False)
fundraising = pd.read_csv(fundraising_wide_path, low_memory=False)
spending = pd.read_csv(spending_wide_path, low_memory=False)

print("Ballot support rows:      ", len(support))
print("Fundraising profile rows: ", len(fundraising))
print("Spending profile rows:    ", len(spending))


Ballot support rows:       98
Fundraising profile rows:  65
Spending profile rows:     76


### A necessary check before merging

Upstream, spending profiles are built **per filing source**, not per candidate.
A candidate who reported through more than one committee therefore appears more
than once, and merging without handling that would silently duplicate their
ballot results.

Profile *shares* cannot simply be added together the way totals can — two
filings each with 50% Mega share do not make 100%. So we combine them as a
**weighted average**, weighted by each filing's dollar total, which is what the
share would have been if all the filings had been one.


In [3]:
spending = spending[spending["candidate_key"].notna()].copy()

# Which candidates filed more than once?
filing_counts = spending.groupby("candidate_key").size()
multi_filers = filing_counts[filing_counts > 1]

print("Candidates with more than one spending filing:", len(multi_filers))

if len(multi_filers) > 0:
    display(
        spending[spending["candidate_key"].isin(multi_filers.index)]
        [["candidate_key", "district", "canonical_candidate", "total_spending"]]
        .sort_values("candidate_key")
    )

    # Combine the filings for each candidate.
    # Totals and counts add up; shares are averaged weighted by total_spending.
    share_columns = [c for c in spending.columns if "_share_" in c]

    combined_rows = []

    for candidate_key, group in spending.groupby("candidate_key"):
        row = {"candidate_key": candidate_key}

        # Totals simply add.
        row["total_spending"] = group["total_spending"].sum()
        row["expenditure_count"] = group["expenditure_count"].sum()

        # Recompute the average from the combined totals.
        if row["expenditure_count"] > 0:
            row["average_expenditure"] = row["total_spending"] / row["expenditure_count"]
        else:
            row["average_expenditure"] = np.nan

        # Median cannot be recovered from summaries, so we do not fake one.
        row["median_expenditure"] = np.nan if len(group) > 1 else group["median_expenditure"].iloc[0]

        # Shares: weighted average by dollars.
        weights = group["total_spending"].to_numpy(dtype=float)
        total_weight = weights.sum()

        for column in share_columns:
            values = group[column].to_numpy(dtype=float)
            if total_weight > 0:
                row[column] = np.nansum(values * weights) / total_weight
            else:
                row[column] = np.nan

        combined_rows.append(row)

    spending = pd.DataFrame(combined_rows)
    print("Spending rows after combining:", len(spending))


Candidates with more than one spending filing: 0


In [4]:
# Keep only the spending columns we actually analyse, so the merge stays readable.
spending_keep = ["candidate_key", "total_spending", "expenditure_count", "average_expenditure", "median_expenditure"]
spending_keep = spending_keep + [c for c in spending.columns if "_share_" in c]
spending_keep = [c for c in spending_keep if c in spending.columns]

analysis = support.merge(
    fundraising,
    on=["year", "district", "candidate_key"],
    how="left",
    suffixes=("", "_fundraising"),
    validate="one_to_one",
)

analysis = analysis.merge(
    spending[spending_keep],
    on="candidate_key",
    how="left",
    validate="one_to_one",
)

print("Candidates in the analysis:", len(analysis))


Candidates in the analysis: 98


## 3. Choose a small, interpretable feature set

We deliberately do **not** throw every available column at the problem. With
only 60-odd candidates, testing dozens of features guarantees that a few look
impressive by chance alone. A short list chosen for interpretability is more
honest and easier to write about.

The cell below also reports how many candidates actually have each feature. A
feature available for only 20 candidates deserves much more caution than one
available for 60.


In [5]:
fundraising_features = [
    "total_contribution_count",
    "average_contribution",
    "median_contribution",
    "amount_share_micro",
    "amount_share_small",
    "amount_share_medium",
    "amount_share_large",
    "amount_share_mega",
    "contribution_share_micro",
    "contribution_share_small",
    "contribution_share_medium",
    "contribution_share_large",
    "contribution_share_mega",
]

spending_features = [
    "expenditure_count",
    "average_expenditure",
    "median_expenditure",
    "spending_amount_share_micro",
    "spending_amount_share_small",
    "spending_amount_share_medium",
    "spending_amount_share_large",
    "spending_amount_share_mega",
    "spending_count_share_micro",
    "spending_count_share_small",
    "spending_count_share_medium",
    "spending_count_share_large",
    "spending_count_share_mega",
]

# Keep only features that really exist, and warn loudly about any that do not,
# so a renamed column upstream cannot silently shrink the analysis.
wanted_features = fundraising_features + spending_features

features = []
missing_features = []

for feature in wanted_features:
    if feature in analysis.columns:
        features.append(feature)
    else:
        missing_features.append(feature)

print("Features available:", len(features))

if len(missing_features) > 0:
    print()
    print("WARNING - these expected features were not found:")
    for feature in missing_features:
        print("   ", feature)
    print("Check the column names in the wide profile files before trusting the results below.")

coverage = pd.DataFrame({
    "feature": features,
    "candidates_with_data": [analysis[f].notna().sum() for f in features],
})

display(coverage)


Features available: 26


,feature,candidates_with_data
0,total_contribution_count,65
1,average_contribution,65
2,median_contribution,65
3,amount_share_micro,65
4,amount_share_small,65
5,amount_share_medium,65
6,amount_share_large,65
7,amount_share_mega,65
8,contribution_share_micro,65
9,contribution_share_small,65


### Method note: profile shares are compositional

The five amount shares add up to 1, and so do the five record shares. That has a
consequence we must keep in mind for every result below: **if one share is
larger, another must be smaller.** They are not independent dials.

So a positive coefficient on `amount_share_micro` does not mean "raising the
Micro share causes more votes." It means "candidates whose money leaned Micro
tended to do better" — which could equally be described as "candidates who
leaned Mega tended to do worse." Read these as descriptions of **candidate
types**, and phrase them that way in the report.


## 4. Correlations with electoral performance

We use the raw outcomes the workplan names — ballot mentions and first-place
votes.

With this many feature-outcome pairs, some correlations will look notable by
chance. That is why the next section screens with regressions and section 6
makes us look at the actual candidates: three filters instead of one.


In [6]:
outcomes = ["mentions", "first_place_votes"]

correlation_rows = []

for feature in features:
    for outcome in outcomes:
        pair = analysis[[feature, outcome]].dropna()

        # Skip pairs with too little variation to be meaningful.
        if len(pair) < 4 or pair[feature].nunique() < 2:
            continue

        correlation_rows.append({
            "feature": feature,
            "outcome": outcome,
            "n": len(pair),
            "pearson": pair[feature].corr(pair[outcome], method="pearson"),
            "spearman": pair[feature].corr(pair[outcome], method="spearman"),
        })

correlations = pd.DataFrame(correlation_rows)

# Show the strongest relationships in either direction, per outcome.
correlations["absolute_pearson"] = correlations["pearson"].abs()

top_correlations = (
    correlations
    .sort_values(["outcome", "absolute_pearson"], ascending=[True, False])
    .groupby("outcome")
    .head(8)
    .drop(columns="absolute_pearson")
)

display(top_correlations.round(3))


,feature,outcome,n,pearson,spearman
1,total_contribution_count,first_place_votes,65,0.679,0.714
29,average_expenditure,first_place_votes,65,0.548,0.648
27,expenditure_count,first_place_votes,65,0.515,0.695
41,spending_amount_share_mega,first_place_votes,65,0.498,0.638
51,spending_count_share_mega,first_place_votes,65,0.490,0.664
39,spending_amount_share_large,first_place_votes,65,-0.467,-0.591
37,spending_amount_share_medium,first_place_votes,65,-0.372,-0.548
33,spending_amount_share_micro,first_place_votes,65,-0.326,-0.569
0,total_contribution_count,mentions,65,0.725,0.755
40,spending_amount_share_mega,mentions,65,0.598,0.658


### Reading checkpoint

Look at **direction and consistency** before size. A feature is genuinely
interesting when:

- it relates to **both** mentions and first-place votes, not just one;
- Pearson and Spearman agree, so the pattern is not created by one extreme
  candidate;
- it survives looking at the actual candidates in section 6.

A large correlation that appears for only one outcome, disagrees between Pearson
and Spearman, and vanishes when you inspect the scatter is almost certainly
noise. Say so in the findings log rather than quietly dropping it — a
well-documented dead end saves the next person the same detour.


## 5. Simple one-feature regressions

For each feature we fit `outcome = a + b × feature` and record R².

This is a **screening step**, not a predictive model. We are ranking features by
how much they might be worth investigating, not building something to forecast
elections.


In [7]:
def simple_ols(data, x, y):
    """Fit one univariate regression and return its main numbers."""
    pair = data[[x, y]].dropna()

    if len(pair) < 4 or pair[x].nunique() < 2 or pair[y].nunique() < 2:
        return None

    x_values = pair[x].to_numpy(dtype=float)
    y_values = pair[y].to_numpy(dtype=float)

    model = sm.OLS(y_values, sm.add_constant(x_values)).fit()

    return {
        "feature": x,
        "outcome": y,
        "n": int(model.nobs),
        "slope": model.params[1],
        "p_value": model.pvalues[1],
        "r_squared": model.rsquared,
    }


regression_rows = []

for feature in features:
    for outcome in outcomes:
        result = simple_ols(analysis, feature, outcome)

        if result is not None:
            regression_rows.append(result)

regressions = pd.DataFrame(regression_rows)

top_models = (
    regressions
    .sort_values(["outcome", "r_squared"], ascending=[True, False])
    .groupby("outcome")
    .head(8)
)

display(top_models.round({"slope": 4, "p_value": 6, "r_squared": 3}))


,feature,outcome,n,slope,p_value,r_squared
1,total_contribution_count,first_place_votes,65,5.4967,0.000000,0.461
29,average_expenditure,first_place_votes,65,7.1746,0.000002,0.300
27,expenditure_count,first_place_votes,65,10.2726,0.000011,0.265
41,spending_amount_share_mega,first_place_votes,65,7437.0353,0.000024,0.248
51,spending_count_share_mega,first_place_votes,65,34033.5513,0.000035,0.240
39,spending_amount_share_large,first_place_votes,65,-10538.9403,0.000088,0.218
37,spending_amount_share_medium,first_place_votes,65,-21008.1507,0.002305,0.138
33,spending_amount_share_micro,first_place_votes,65,-87949.7122,0.008141,0.106
0,total_contribution_count,mentions,65,15.9679,0.000000,0.525
40,spending_amount_share_mega,mentions,65,24380.4372,0.000000,0.358


In [8]:
fig = px.bar(
    top_models,
    x="r_squared",
    y="feature",
    color="outcome",
    facet_col="outcome",
    orientation="h",
    hover_data=["n", "slope", "p_value"],
    labels={
        "r_squared": "R-squared",
        "feature": "Finance feature",
        "outcome": "Electoral outcome",
    },
    title="Which fine-grained finance features explain the most variation?",
)

# Let each panel show its own feature ordering.
fig.update_yaxes(matches=None)
fig.show()


### An important comparison to make

Look up the R² values that **total** fundraising and spending achieved in
notebook 10, and compare them with the best fine-grained features here.

If a fine-grained feature beats the totals, structure carries information beyond
scale — a real finding. If none of them do, the honest conclusion is that
**total money is the main story and composition adds little**, which is equally
worth reporting and saves effort later.

Watch for one trap: `total_contribution_count` is partly a measure of *size*,
not structure. A campaign with more donors usually raised more money. If it
tops the table, that is closer to confirming notebook 10 than to discovering
something new about composition.


## 6. Interactive feature explorer

The main qualitative tool in this notebook. **Edit the two variables at the top
of the cell and rerun** to explore any pairing.

Use the tables above to pick a feature, then use this figure to ask: which
candidates create the relationship? Which contradict it? Is one district driving
it? Are the outliers viable or not?

This is where a coefficient starts becoming a political story.


In [9]:
# ------------------------------------------------------------
# EDIT THESE TWO LINES to explore a different relationship.
# ------------------------------------------------------------
x_variable = "contribution_share_micro"
y_variable = "mentions"
# ------------------------------------------------------------

plot_data = analysis.dropna(subset=[x_variable, y_variable]).copy()
plot_data["district"] = plot_data["district"].astype(str)

fig = px.scatter(
    plot_data,
    x=x_variable,
    y=y_variable,
    color="district",
    symbol="is_viable",
    hover_name="canonical_candidate",
    hover_data={
        "district": True,
        x_variable: ":.3f",
        "mentions": ":,.0f",
        "first_place_votes": ":,.0f",
        "is_viable": True,
    },
    labels={
        "district": "District",
        "is_viable": "Viable",
    },
    title=f"{x_variable} vs. {y_variable}",
)

fig.show()


## 7. Fundraising composition, candidate by candidate

Sometimes the clearest way to understand a candidate is to see their whole
funding mix at once rather than one share at a time. Each bar is a candidate,
split by where their dollars came from.

**Edit the district and rerun.** Look for candidates whose bar looks unlike
their neighbours', and check whether their electoral result was unusual too.


In [10]:
# ------------------------------------------------------------
# EDIT THIS LINE to look at a different district.
# ------------------------------------------------------------
district_to_explore = 3
# ------------------------------------------------------------

amount_share_columns = [
    "amount_share_micro",
    "amount_share_small",
    "amount_share_medium",
    "amount_share_large",
    "amount_share_mega",
]

available_columns = [c for c in amount_share_columns if c in analysis.columns]

district_data = analysis[analysis["district"] == district_to_explore]

# melt turns one row per candidate into one row per candidate-and-bin,
# which is the shape plotly needs for a stacked bar chart.
profile_plot = district_data[
    ["canonical_candidate", "is_viable", "mentions"] + available_columns
].melt(
    id_vars=["canonical_candidate", "is_viable", "mentions"],
    var_name="bin",
    value_name="amount_share",
)

# Tidy up the bin names for the legend.
profile_plot["bin"] = (
    profile_plot["bin"].str.replace("amount_share_", "", regex=False).str.title()
)

fig = px.bar(
    profile_plot,
    x="canonical_candidate",
    y="amount_share",
    color="bin",
    hover_data=["is_viable", "mentions"],
    labels={
        "canonical_candidate": "Candidate",
        "amount_share": "Share of fundraising dollars",
        "bin": "Contribution size",
    },
    title=f"District {district_to_explore} - fundraising composition by candidate",
)

fig.update_layout(xaxis_tickangle=-45)
fig.show()


## 8. Spending composition, candidate by candidate

The same view for expenditure size. Remember the caveat: these bins are
transaction **size**, not purpose, so a large bar means large individual
payments, not necessarily advertising.


In [11]:
# ------------------------------------------------------------
# EDIT THIS LINE to look at a different district.
# ------------------------------------------------------------
district_to_explore = 3
# ------------------------------------------------------------

spending_share_columns = [
    "spending_amount_share_micro",
    "spending_amount_share_small",
    "spending_amount_share_medium",
    "spending_amount_share_large",
    "spending_amount_share_mega",
]

available_columns = [c for c in spending_share_columns if c in analysis.columns]

if len(available_columns) == 0:
    print("No spending share columns available - skipping this figure.")
else:
    district_data = analysis[analysis["district"] == district_to_explore]

    spending_plot = district_data[
        ["canonical_candidate", "is_viable", "mentions"] + available_columns
    ].melt(
        id_vars=["canonical_candidate", "is_viable", "mentions"],
        var_name="bin",
        value_name="spending_share",
    )

    spending_plot["bin"] = (
        spending_plot["bin"]
        .str.replace("spending_amount_share_", "", regex=False)
        .str.title()
    )

    fig = px.bar(
        spending_plot,
        x="canonical_candidate",
        y="spending_share",
        color="bin",
        hover_data=["is_viable", "mentions"],
        labels={
            "canonical_candidate": "Candidate",
            "spending_share": "Share of reported spending",
            "bin": "Expenditure size",
        },
        title=f"District {district_to_explore} - spending composition by candidate",
    )

    fig.update_layout(xaxis_tickangle=-45)
    fig.show()


## 9. Findings log — fine-grained pass

Record answers, not just coefficients.

**Which dimensions matter?**

- (The 3 to 5 features with the strongest relationship to mentions.)
- (Are they the same features for first-place votes? Disagreement is itself
  informative.)
- (Does contribution **count** look more informative than average contribution
  **size**? If so, the story is about breadth of support rather than depth of
  pockets.)

**Do they beat the totals?**

- (Compare against notebook 10's R² values. If composition adds nothing beyond
  scale, say so plainly — that is a real result.)

**Candidate types**

- (Do viable candidates share a distinctive contribution-size mix?)
- (Any pair of candidates with **similar total** fundraising but very different
  **composition** and very different results? Those pairs are the best possible
  evidence that structure matters, and the best material for the report.)

**Dead ends**

- (Features that looked promising but did not survive inspection. Worth
  recording so nobody repeats the detour.)

**Interpretive rule to keep**

Because shares sum to one, never write "increasing Micro share causes more
votes." Write "candidates whose funding leaned Micro tended to receive more
mentions."

**Next**

- Notebook 12 identifies which candidates break the overall pattern and builds
  the shortlist for qualitative research.
- These five bins are the dimensions for the later profile-similarity work, so
  note here which ones actually separate candidates in practice.


## 10. Export the screening tables

In [12]:
output_dir = PROCESSED / "finance_analysis" / str(YEAR) / "question_3" / "fine_grained"
output_dir.mkdir(parents=True, exist_ok=True)

correlations.to_csv(output_dir / "fine_grained_correlations.csv", index=False)
regressions.to_csv(output_dir / "fine_grained_univariate_ols.csv", index=False)
top_correlations.to_csv(output_dir / "top_fine_grained_correlations.csv", index=False)
top_models.to_csv(output_dir / "top_fine_grained_models.csv", index=False)
coverage.to_csv(output_dir / "feature_coverage.csv", index=False)

print("SAVED:", output_dir)


SAVED: /Users/marcy/mggg/projects/Portland/portland-fundraising-support-analysis/data/processed/finance_analysis/2024/question_3/fine_grained
